# HW1 Part 3 - Training a Classifier

Here, you will use your clean corpus of restaurant reviews from Part 1 and the tokenizer you trained in Part 2 to train a model. You will create and train a small transformer to predict whether a review is positive or negative, then measure how well it does.

A GPU will make training faster, but the model is small enough to train on CPU. To use one in Colab, set **Runtime > Change runtime type > T4 GPU**.

Submit this notebook run top to bottom, with all outputs visible.

In [1]:
!pip install -q tokenizers

In [2]:
import random
import sys

import torch
import torch.nn as nn

sys.path.insert(0, ".")
torch.manual_seed(1990)
random.seed(1990)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

device: cpu


## 3.0 - Load the corpus and the tokenizer

`build_pipeline` should return the cleaned records, and `train_bpe` should turn their text into a tokenizer.

Make sure you train this tokenizer on **the whole review corpus**, NOT on the built-in sentences from Part 2.

In [3]:
from part1_data_processing import DEFAULT_THRESHOLDS, build_pipeline, load_corpus
from part2_tokenizing import train_bpe

VOCAB_SIZE = 2000

def load_reviews() -> list[dict]:
    """
    In:  nothing
    Out: the cleaned records from your Part 1 pipeline
    """
    return build_pipeline(DEFAULT_THRESHOLDS).run(load_corpus())


def load_tokenizer(texts: list[str], vocab_size: int = VOCAB_SIZE):
    """
    In:  the cleaned review texts and a vocabulary size
    Out: a BPE tokenizer trained on them, using your Part 2 train_bpe
    """
    return train_bpe(texts, vocab_size)

## 3.1 - Make a train and test split

The reviews carry a 1-5 star rating, which becomes a binary label. Treat 1 and 2 stars as negative and 4 and 5 as positive. Drop the 3-star reviews, as they are neutral. Keep the two classes balanced in both splits.

  shuffling: https://docs.python.org/3/library/random.html#random.shuffle

In [4]:
def make_splits(records: list[dict], test_frac: float = 0.2) -> tuple[list, list]:
    """
    In:  the cleaned records, and the fraction to hold out for testing
    Out: (train, test), each a list of (text, label) pairs, where label is
         0 for negative and 1 for positive

    Drop the 3-star reviews. Balance the two classes, and shuffle before
    splitting so the split does not follow the order of the corpus.
    """
    neg = [(r["text"], 0) for r in records if r["label"] in (1, 2)]
    pos = [(r["text"], 1) for r in records if r["label"] in (4, 5)]
    random.shuffle(neg)
    random.shuffle(pos)

    n = min(len(neg), len(pos))
    neg = neg[:n]
    pos = pos[:n]

    n_test = int(n * test_frac)
    test = neg[:n_test] + pos[:n_test]
    train = neg[n_test:] + pos[n_test:]
    random.shuffle(train)
    random.shuffle(test)
    return train, test

## 3.2 - Train the model

Build a simple transformer model. The model embeds each token, lets the tokens attend to each other once, averages the result, and maps that average to two scores. 

Reviews differ in length, so `encode_batch` below pads them to a fixed length and returns a mask marking which positions are real. Use the mask twice: attention should ignore padded positions, and the average at the end should be taken over real tokens only.

  modules:   https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html

  embedding: https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html

  attention: https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
  
  training:  https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html

In [5]:
MAX_LEN = 256


def encode_batch(tokenizer, texts: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Tokenizes and pads a batch. Written for you.

    In:  a tokenizer and a list of strings
    Out: (ids, mask), both LongTensors shaped (len(texts), n), where n is the
         longest tokenized review in the batch, capped at MAX_LEN. mask is 1 at
         real token positions and 0 at padding.
    """
    rows = [tokenizer.encode(t).ids[:MAX_LEN] for t in texts]
    width = max(len(r) for r in rows)
    ids = torch.zeros(len(rows), width, dtype=torch.long)
    mask = torch.zeros(len(rows), width, dtype=torch.long)
    for i, row in enumerate(rows):
        ids[i, :len(row)] = torch.tensor(row, dtype=torch.long)
        mask[i, :len(row)] = 1
    return ids.to(DEVICE), mask.to(DEVICE)


class ReviewClassifier(nn.Module):
    """
    A token embedding, a position embedding, one self-attention block, a mean
    over the real tokens, and a linear layer to two classes.
    """

    def __init__(self, vocab_size: int, d_model: int = 64, n_heads: int = 4):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(MAX_LEN, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Linear(d_model * 2, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, 2)

    def forward(self, ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """
        In:  ids and mask, both (batch, length)
        Out: logits of shape (batch, 2)
        """
        _, seq_len = ids.shape
        pos = torch.arange(seq_len, device=ids.device).unsqueeze(0)
        x = self.tok_emb(ids) + self.pos_emb(pos)
        key_padding_mask = (mask == 0)
        attn_out, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        mask_f = mask.unsqueeze(-1).float()
        pooled = (x * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
        return self.fc(pooled)


def train(model, train_split: list, tokenizer, epochs: int = 8,
          batch_size: int = 16, lr: float = 1e-3):
    """
    In:  a model, the training split, a tokenizer, and the usual knobs
    Out: the trained model

    Print the average loss each epoch, so you can monitor training progress.
    """
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(1, epochs + 1):
        data = list(train_split)
        random.shuffle(data)
        total_loss = 0.0
        n_batches = 0
        for start in range(0, len(data), batch_size):
            batch = data[start:start + batch_size]
            texts = [t for t, _ in batch]
            labels = torch.tensor([y for _, y in batch], dtype=torch.long, device=DEVICE)
            ids, mask = encode_batch(tokenizer, texts)
            optimizer.zero_grad()
            logits = model(ids, mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item())
            n_batches += 1
        print(f"  epoch {epoch:>2}: loss = {total_loss / max(n_batches, 1):.4f}")
    return model

## 3.3 - Evaluate performance

Return your model's overall accuracy. For a more in-depth evaluation, also return precision and
recall for both classes.

For the positive class, precision is the fraction of reviews the model labelled positive that
really were positive, and recall is the fraction of the positive reviews that it found. The
negative class is the same with the labels swapped.

In [6]:
def evaluate(model, test_split: list, tokenizer) -> dict:
    """
    In:  a trained model, the test split, and a tokenizer
    Out: a dict with keys "accuracy", "precision_negative", "recall_negative",
         "precision_positive", and "recall_positive"
    """
    model.eval()
    tp = tn = fp = fn = 0
    with torch.no_grad():
        for start in range(0, len(test_split), 32):
            batch = test_split[start:start + 32]
            texts = [t for t, _ in batch]
            labels = [y for _, y in batch]
            ids, mask = encode_batch(tokenizer, texts)
            preds = model(ids, mask).argmax(dim=-1).cpu().tolist()
            for p, y in zip(preds, labels):
                if p == 1 and y == 1:
                    tp += 1
                elif p == 0 and y == 0:
                    tn += 1
                elif p == 1 and y == 0:
                    fp += 1
                else:
                    fn += 1
    total = max(tp + tn + fp + fn, 1)
    return {
        "accuracy": (tp + tn) / total,
        "precision_negative": tn / (tn + fn) if (tn + fn) > 0 else 0.0,
        "recall_negative": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "precision_positive": tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        "recall_positive": tp / (tp + fn) if (tp + fn) > 0 else 0.0,
    }

### Run it

Written for you. Runs everything above and prints the results.

In [7]:
records = load_reviews()
print(f"{len(records)} clean reviews")

tokenizer = load_tokenizer([r["text"] for r in records])

train_split, test_split = make_splits(records)
print(f"{len(train_split)} train, {len(test_split)} test")

model = ReviewClassifier(tokenizer.get_vocab_size()).to(DEVICE)
model = train(model, train_split, tokenizer)

results = evaluate(model, test_split, tokenizer)
print()
for name, value in results.items():
    print(f"  {name:<20} {value:.3f}")

770 clean reviews
494 train, 122 test
  epoch  1: loss = 0.6440
  epoch  2: loss = 0.4198
  epoch  3: loss = 0.2623
  epoch  4: loss = 0.1494
  epoch  5: loss = 0.0494
  epoch  6: loss = 0.0163
  epoch  7: loss = 0.0051
  epoch  8: loss = 0.0028

  accuracy             0.852
  precision_negative   0.852
  recall_negative      0.852
  precision_positive   0.852
  recall_positive      0.852
